# 00 · Environment check

What this node can run: RDMA devices, GIDs, libfabric providers, pyverbs, native builds, tests.

In [1]:
MODE = "local"            # "local" or "cluster" (run on each node)
RESULTS_SUBDIR = "env"
NETDEV = None             # 10GbE interface carrying SoftRoCE, e.g. "enp1s0f0"; used only to print the rxe setup command
RUN_TESTS = True

In [2]:
import os, sys, json, socket, subprocess, time
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "middleware").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
HOST = socket.gethostname()
STAMP = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
RESULTS = (ROOT / "benchmarks" / "results" / RESULTS_SUBDIR) if MODE == "cluster" else Path("/tmp/scuffed-local") / RESULTS_SUBDIR
RESULTS.mkdir(parents=True, exist_ok=True)
FIGDIR = ROOT / "docs" / "figures"
(FIGDIR / MODE).mkdir(parents=True, exist_ok=True)

import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 170, "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "legend.fontsize": 7.5, "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25, "lines.linewidth": 1.4,
    "lines.markersize": 4, "figure.constrained_layout.use": False})
C = {"hot": "#f97316", "cold": "#38bdf8", "spill": "#a78bfa", "ok": "#34d399", "bad": "#f43f5e",
     "gold": "#facc15", "ink": "#1f2937", "muted": "#94a3b8"}

def fig(name, model=False, caption=None, f=None):
    """Save the current figure as PNG + PDF and show it.
    model=True: arithmetic or repo results, same on every machine -> docs/figures/<name>
    model=False: measured on this machine -> docs/figures/<MODE>/<name>"""
    f = f or plt.gcf()
    d = FIGDIR if model else FIGDIR / MODE
    f.tight_layout()
    for ext in ("png", "pdf"):
        f.savefig(d / f"{name}.{ext}", bbox_inches="tight")
    print(f"figure -> {d.relative_to(ROOT)}/{name}.png" + (f"  |  {caption}" if caption else ""))
    plt.show()

def save(name, obj):
    p = RESULTS / f"{HOST}_{name}.json"
    obj = {"host": HOST, "utc": STAMP, "mode": MODE, **obj}
    p.write_text(json.dumps(obj, indent=2, default=str))
    print("wrote", p)
    return p

def sh(cmd, check=True):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, cwd=ROOT)
    print(r.stdout[-4000:], r.stderr[-2000:])
    if check and r.returncode:
        raise RuntimeError(f"{cmd} -> {r.returncode}")
    return r
print("repo:", ROOT, "| host:", HOST, "| mode:", MODE, "| results ->", RESULTS, "| figures ->", FIGDIR)

repo: /home/claude/scuffedrdma | host: vm | mode: local | results -> /tmp/scuffed-local/env | figures -> /home/claude/scuffedrdma/docs/figures


Run this first on every node. It records what the node actually has (RDMA devices, GIDs, libfabric providers,
pyverbs, compilers), builds the three native libraries, and runs the test suite. On a node with an RDMA
device the `rdma`-marked tests run too (verbs provider, `verbs;ofi_rxm`, native libibverbs loopback over rxe0).

In [3]:
import platform, shutil
info = {"uname": platform.uname()._asdict(), "python": sys.version, "cpus": os.cpu_count()}
for cmd in ["lscpu | head -20", "nproc", "uname -r"]:
    info[cmd] = sh(cmd, check=False).stdout

$ lscpu | head -20
Architecture:                            x86_64
CPU op-mode(s):                          32-bit, 64-bit
Address sizes:                           46 bits physical, 48 bits virtual
Byte Order:                              Little Endian
CPU(s):                                  2
On-line CPU(s) list:                     0,1
Vendor ID:                               GenuineIntel
Model name:                              Intel(R) Xeon(R) Processor @ 2.80GHz
CPU family:                              6
Model:                                   85
Thread(s) per core:                      1
Core(s) per socket:                      2
Socket(s):                               1
Stepping:                                7
BogoMIPS:                                5600.45
Flags:                                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xtopology nonstop_

## RDMA devices and GIDs

In [4]:
ib = Path("/sys/class/infiniband")
devs = sorted(p.name for p in ib.iterdir()) if ib.is_dir() else []
info["rdma_devices"] = devs
print("RDMA devices:", devs or "none")
if not devs:
    nd = NETDEV or "<10GbE iface>"
    print(f"No device. SoftRoCE: sudo modprobe rdma_rxe && sudo rdma link add rxe0 type rxe netdev {nd}")
gids = []
for d in devs:
    for g in sorted((ib / d / "ports/1/gids").iterdir(), key=lambda p: int(p.name)):
        val = g.read_text().strip()
        if val.strip("0:") == "":
            continue
        try:
            typ = (ib / d / "ports/1/gid_attrs/types" / g.name).read_text().strip()
        except OSError:
            typ = "?"
        gids.append({"dev": d, "index": int(g.name), "gid": val, "type": typ})
info["gids"] = gids
pd_gids = __import__("pandas").DataFrame(gids) if gids else None
pd_gids

RDMA devices: none
No device. SoftRoCE: sudo modprobe rdma_rxe && sudo rdma link add rxe0 type rxe netdev <10GbE iface>


In [5]:
for cmd in ["ibv_devices", "ibv_devinfo -v | head -60", "rdma link show"]:
    if shutil.which(cmd.split()[0]):
        info[cmd] = sh(cmd, check=False).stdout

$ ibv_devices
 Failed to get IB devices list: Function not implemented

$ ibv_devinfo -v | head -60
 Failed to get IB devices list: Function not implemented

$ rdma link show
 Failed to open NETLINK_RDMA socket



## libfabric providers and pyverbs

In [6]:
if shutil.which("fi_info"):
    info["fi_info_version"] = sh("fi_info --version", check=False).stdout
    info["fi_providers"] = [l.strip(" :") for l in sh("fi_info -l", check=False).stdout.split("\n") if l and not l.startswith(" ")]
    for p in ["verbs", "verbs;ofi_rxm", "tcp", "shm"]:
        r = sh(f"fi_info -p '{p}' -c 'FI_RMA|FI_MSG' 2>/dev/null | grep -E 'provider|type|domain' | head -6", check=False)
        info[f"fi_info[{p}]"] = r.stdout
else:
    print("libfabric missing: sudo apt install libfabric-dev libfabric-bin")
try:
    import pyverbs.device as d
    info["pyverbs"] = [x.name.decode() for x in d.get_device_list()]
except Exception as e:
    info["pyverbs"] = f"unavailable: {e}"
print("pyverbs:", info["pyverbs"])

$ fi_info --version


fi_info: 1.17.0
libfabric: 1.17.0
libfabric api: 1.17
 
$ fi_info -l


psm2:
    version: 117.0
psm:
    version: 117.0
ofi_rxm:
    version: 117.0
ofi_rxd:
    version: 117.0
shm:
    version: 117.0
udp:
    version: 117.0
tcp:
    version: 117.0
sockets:
    version: 117.0
net:
    version: 117.0
ofi_hook_perf:
    version: 117.0
ofi_hook_debug:
    version: 117.0
ofi_hook_noop:
    version: 117.0
ofi_hook_hmem:
    version: 117.0
ofi_hook_dmabuf_peer_mem:
    version: 117.0
ofi_mrail:
    version: 117.0
 
$ fi_info -p 'verbs' -c 'FI_RMA|FI_MSG' 2>/dev/null | grep -E 'provider|type|domain' | head -6


 
$ fi_info -p 'verbs;ofi_rxm' -c 'FI_RMA|FI_MSG' 2>/dev/null | grep -E 'provider|type|domain' | head -6


 
$ fi_info -p 'tcp' -c 'FI_RMA|FI_MSG' 2>/dev/null | grep -E 'provider|type|domain' | head -6


provider: tcp;ofi_rxm
    domain: eth0
    type: FI_EP_RDM
provider: tcp;ofi_rxm
    domain: lo
    type: FI_EP_RDM
 
$ fi_info -p 'shm' -c 'FI_RMA|FI_MSG' 2>/dev/null | grep -E 'provider|type|domain' | head -6


provider: shm
    domain: shm
    type: FI_EP_RDM
provider: shm
    domain: shm
    type: FI_EP_RDM
 
pyverbs: unavailable: No module named 'pyverbs.device'


## Build the native libraries

In [7]:
for d in ["middleware/libscuffedfabric", "middleware/libscuffedverbs", "middleware/nccl_plugin"]:
    r = sh(f"make -C {d}", check=False)
    info[f"build:{d}"] = "ok" if r.returncode == 0 else r.stderr[-1500:]
    print(d, info[f"build:{d}"][:200])

$ make -C middleware/libscuffedfabric
make: Entering directory '/home/claude/scuffedrdma/middleware/libscuffedfabric'
make: Nothing to be done for 'all'.
make: Leaving directory '/home/claude/scuffedrdma/middleware/libscuffedfabric'
 
middleware/libscuffedfabric ok
$ make -C middleware/libscuffedverbs
make: Entering directory '/home/claude/scuffedrdma/middleware/libscuffedverbs'
make: Nothing to be done for 'all'.
make: Leaving directory '/home/claude/scuffedrdma/middleware/libscuffedverbs'
 
middleware/libscuffedverbs ok
$ make -C middleware/nccl_plugin
make: Entering directory '/home/claude/scuffedrdma/middleware/nccl_plugin'
make: Nothing to be done for 'all'.
make: Leaving directory '/home/claude/scuffedrdma/middleware/nccl_plugin'
 
middleware/nccl_plugin ok


## Test suite

In [8]:
if RUN_TESTS:
    r = sh("python3 -m pytest -q -p no:cacheprovider", check=False)
    info["pytest"] = r.stdout[-3000:]
    print("pytest exit", r.returncode)

$ python3 -m pytest -q -p no:cacheprovider


........................sss.sssssss..................................... [ 46%]
...................................................ss................... [ 92%]
..........ss                                                             [100%]
 
pytest exit 0


In [9]:
save("environment", info)

wrote /tmp/scuffed-local/env/vm_environment.json


PosixPath('/tmp/scuffed-local/env/vm_environment.json')